In [9]:
from itertools import product
import pandas as pd
import cobra
import json

import sys
sys.path.insert(1, '../../scripts')
from utils.load_environmental_variables import build_files_path
prebuild = '/data2/hratch/human_me/prebuild/'

Dictionary of all M-metabolites used in expression module

In [3]:
nucleotides = ['a', 'c', 'g', 'u']
nps = ['mp', 'tp', 'dp']
atop_hydrolysis = ['atp', 'adp', 'h', 'pi', 'h2o']

aa_s = ['ala_L', 'arg_L', 'asn_L', 'asp_L', 'cys_L', 'gln_L', 'glu_L', 'gly', 'his_L','ile_L','leu_L','lys_L','phe_L',
        'pro_L','ser_L', 'thr_L','trp_L','tyr_L','val_L', 'met_L']


required_metabolites = {'c': [], 'l': [], 'm': [], 'r': [], 'e': [], 'x': [], 'n': [], 'g': [], 'i': [], 'pm': []}

required_metabolites['c'] += ['ppi_c', 'amet_c', 'ahcys_c', 'g6p_c', 'chsterol_c', 
                             'clpn_hs_c', 'pail_hs_c', 'pchol_hs_c','pe_hs_c','pglyc_hs_c','ps_hs_c',
                              'sphmyln_hs_c']

required_metabolites['n'] += ['ppi_n', 'amet_n', 'ahcys_n', 'adp_n', 'gdp_n']

required_metabolites['r'] += ['h2o2_r', 'hdca_r', 'gpi_hs_r', 'o2_r', 'udpacgal_r', 
                              'udpgal_r', 'uacgam_r', 'udp_r'] 
                              #'gpi_sig[r', 'm_em_3gacpail_hs[r', 'm_em_3gacpail_prot_hs[r', 'pre_prot[r']
required_metabolites['g'] += ['udpacgal_g', 'udpgal_g', 'uacgam_g', 'h_g', 'udp_g']
required_metabolites['l'] += ['o2_l', 'h2o2_l', 'udpacgal_l', 'udp_l', 'hdca_l']


# add nucleotides
required_metabolites['c'] += [''.join(k) + '_c' for k in list(product(nucleotides, nps))]
required_metabolites['n'] += [n + 'tp_n' for n in nucleotides] + [n + 'mp_n' for n in nucleotides] + ['cdp_n']
required_metabolites['n'] += ['d' + n + 'tp_n' for n in nucleotides][:-1] + ['dttp_n']
# add amino acids
for comp in ['c', 'm', 'l', 'x', 'n', 'r']:
    required_metabolites[comp] += [a + '_' + comp for a in aa_s]

# add atp hydrolysis metabolites
for comp in ['m', 'x', 'r', 'l', 'c', 'n']:
    required_metabolites[comp] += [hydro + '_' + comp for hydro in atop_hydrolysis]

required_metabolites = {k: sorted(set(v)) for k, v in required_metabolites.items()}

In [6]:
json.dump(required_metabolites, open(build_files_path + "required_metabolic_model_metabolites.json", 'w' ))

Generate dataframe of relevant information for metabolites

In [11]:
model = cobra.io.read_sbml_model(prebuild + 'recon2_2.xml')

In [12]:
def bool_metabolite(m_id, compartment):
    try: 
        m = model.metabolites.get_by_id(m_id + '_' + compartment)
        return True, m 
    except:
        return False, None

In [13]:
metab_ids = [item for sublist in list(required_metabolites.values()) for item in sublist]
metab_ids = sorted(set(['_'.join(i.split('_')[:-1]) for i in metab_ids]))

counter = 0
compartments = sorted(required_metabolites.keys())
rmd = pd.DataFrame(columns = ['id', 'name', 'charge', 'elements', 'formula'])

for m_id in metab_ids:
    m_found = False
    counter_ = 0
    while not m_found: # find the first compartment the metabolite is in and get the information
#         print(counter_)
        compartment = compartments[counter_]
        m_found, m = bool_metabolite(m_id, compartment)
        counter_ += 1
        if counter_ > len(compartments):
            raise ValueError('metabolite note found')
    rmd.loc[counter,:] = [m_id, m.name, m.charge, m.elements, m.formula]
    counter += 1

rmd.index = rmd.id
rmd.drop(columns = ['id'], inplace = True)
rmd.charge = rmd.charge.astype(int)
rmd.to_csv(build_files_path + 'required_metabolic_model_metabolites.csv')